In [ ]:
import os
import shutil


def download_stockfish():
    stockfish_dir = "/kaggle/working/stockfish/"
    stockfish_binary = os.path.join(stockfish_dir, "stockfish")
    source_path = "/kaggle/input/stockfish/other/binary/1/stockfish"

    os.makedirs(stockfish_dir, exist_ok=True)

    try:
        if not os.path.exists(stockfish_binary):
            shutil.copy2(source_path, stockfish_binary)
            os.chmod(stockfish_binary, 0o755)
        return stockfish_binary
    except Exception as e:
        print(f"Error setting up Stockfish: {e}")
        return None


STOCKFISH_PATH = download_stockfish()
print(f"Stockfish path: {STOCKFISH_PATH}")

In [ ]:
from dataclasses import dataclass


@dataclass
class TrainingConfig:
    hidden_size: int = 256
    intermediate_size: int = 1024
    num_hidden_layers: int = 6
    num_attention_heads: int = 8
    max_position_embeddings: int = 512
    max_length: int = 512

    batch_size: int = 96
    learning_rate: float = 3e-4
    weight_decay: float = 0.01

    pretrain_epochs: int = 2
    instruct_supervised_epochs: int = 2
    instruct_stockfish_epochs: int = 2

    max_games: int = 200
    train_split: float = 0.9

    stockfish_path: str = ""
    stockfish_depth: int = 10
    stockfish_temperature: float = 50.0
    use_stockfish_every_n_batches: int = 5
    stockfish_alpha: float = 0.3
    invalid_move_penalty: float = 5.0

    log_every_n_batches: int = 50

    save_dir: str = "chess_model"


config = TrainingConfig()
config.stockfish_path = STOCKFISH_PATH  # type: ignore
print("Configuration loaded:")
print(
    f"  Model size: {config.num_hidden_layers} layers, {config.hidden_size} hidden dim"
)
print(
    f"  Training: {config.pretrain_epochs + config.instruct_supervised_epochs + config.instruct_stockfish_epochs} total epochs"
)
print(f"  Max games: {config.max_games}")
print(f"  Stockfish: {config.stockfish_path}")

In [ ]:
from typing import List, Optional
import json


class ChessTokenizer:
    def __init__(self, instruct_mode: bool = False):
        self.instruct_mode = instruct_mode

        base_special = ["<PAD>", "<BOS>", "<EOS>", "<UNK>"]

        if instruct_mode:
            self.special_tokens = base_special + ["<|white|>", "<|black|>", "<|end|>"]
        else:
            self.special_tokens = base_special

        self.vocab = self._build_vocab()
        self.token_to_id = {token: idx for idx, token in enumerate(self.vocab)}
        self.id_to_token = {idx: token for token, idx in self.token_to_id.items()}

        self.pad_token = "<PAD>"
        self.bos_token = "<BOS>"
        self.eos_token = "<EOS>"
        self.unk_token = "<UNK>"

        self.pad_token_id = self.token_to_id[self.pad_token]
        self.bos_token_id = self.token_to_id[self.bos_token]
        self.eos_token_id = self.token_to_id[self.eos_token]
        self.unk_token_id = self.token_to_id[self.unk_token]

        if instruct_mode:
            self.white_token = "<|white|>"
            self.black_token = "<|black|>"
            self.end_token = "<|end|>"
            self.white_token_id = self.token_to_id[self.white_token]
            self.black_token_id = self.token_to_id[self.black_token]
            self.end_token_id = self.token_to_id[self.end_token]

    def _build_vocab(self) -> List[str]:
        vocab = []
        vocab.extend(self.special_tokens)

        files = "abcdefgh"
        ranks = "12345678"
        squares = [f + r for f in files for r in ranks]

        for from_sq in squares:
            for to_sq in squares:
                vocab.append(f"{from_sq}{to_sq}")

                if from_sq[1] == "7" and to_sq[1] == "8":
                    for piece in ["q", "r", "b", "n"]:
                        vocab.append(f"{from_sq}{to_sq}{piece}")

                if from_sq[1] == "2" and to_sq[1] == "1":
                    for piece in ["q", "r", "b", "n"]:
                        vocab.append(f"{from_sq}{to_sq}{piece}")

        return vocab

    def encode_move(self, move_uci: str) -> str:
        return move_uci

    def decode_move(self, move_token: str) -> Optional[str]:
        if move_token in self.token_to_id and len(move_token) >= 4:
            return move_token
        return None

    def encode_pretrain(self, moves: List[str]) -> List[int]:
        tokens = [self.bos_token]
        tokens.extend(moves)
        tokens.append(self.eos_token)

        return [self.token_to_id.get(token, self.unk_token_id) for token in tokens]

    def encode_instruct(
        self, moves: List[str], include_last_black: bool = True
    ) -> List[int]:
        tokens = [self.bos_token]

        for i, move in enumerate(moves):
            if i % 2 == 0:
                tokens.append(self.white_token)
            else:
                tokens.append(self.black_token)

            tokens.append(move)
            tokens.append(self.end_token)

        if not include_last_black and len(moves) % 2 == 0:
            tokens = tokens[:-3]

        tokens.append(self.eos_token)

        return [self.token_to_id.get(token, self.unk_token_id) for token in tokens]

    def decode(self, token_ids: List[int]) -> List[str]:
        return [self.id_to_token.get(id, self.unk_token) for id in token_ids]

    def __len__(self):
        return len(self.vocab)

    def save_pretrained(self, path: str):
        os.makedirs(path, exist_ok=True)

        vocab_data = {
            "vocab": self.vocab,
            "special_tokens": self.special_tokens,
            "instruct_mode": self.instruct_mode,
        }

        with open(os.path.join(path, "vocab.json"), "w") as f:
            json.dump(vocab_data, f, indent=2)

        config = {
            "pad_token": self.pad_token,
            "bos_token": self.bos_token,
            "eos_token": self.eos_token,
            "unk_token": self.unk_token,
            "instruct_mode": self.instruct_mode,
        }

        if self.instruct_mode:
            config["white_token"] = self.white_token
            config["black_token"] = self.black_token
            config["end_token"] = self.end_token

        with open(os.path.join(path, "tokenizer_config.json"), "w") as f:
            json.dump(config, f, indent=2)


pretrain_tokenizer = ChessTokenizer(instruct_mode=False)
instruct_tokenizer = ChessTokenizer(instruct_mode=True)

print(f"Pretrain tokenizer vocab size: {len(pretrain_tokenizer)}")
print(f"Instruct tokenizer vocab size: {len(instruct_tokenizer)}")
print(
    f"\nExample pretrain encoding: {pretrain_tokenizer.encode_pretrain(['e2e4', 'e7e5'])}"
)
print(
    f"Example instruct encoding: {instruct_tokenizer.encode_instruct(['e2e4', 'e7e5'])}"
)

In [ ]:
from torch.utils.data import Dataset
from typing import Dict
import torch


class ChessPretrainDataset(Dataset):
    def __init__(
        self, samples: List[Dict], tokenizer: ChessTokenizer, max_length: int = 512
    ):
        self.samples = samples
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]
        moves = sample["moves"]

        input_ids = self.tokenizer.encode_pretrain(moves)

        if len(input_ids) > self.max_length:
            input_ids = input_ids[: self.max_length]

        labels = input_ids[1:] + [self.tokenizer.pad_token_id]

        input_ids = input_ids + [self.tokenizer.pad_token_id] * (
            self.max_length - len(input_ids)
        )
        labels = labels + [-100] * (self.max_length - len(labels))

        attention_mask = [
            1 if id != self.tokenizer.pad_token_id else 0 for id in input_ids
        ]

        return {
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
            "labels": torch.tensor(labels, dtype=torch.long),
        }


class ChessInstructSupervisedDataset(Dataset):
    def __init__(
        self, samples: List[Dict], tokenizer: ChessTokenizer, max_length: int = 512
    ):
        self.samples = samples
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]
        moves = sample["moves"]

        full_input_ids = self.tokenizer.encode_instruct(moves, include_last_black=True)

        if len(full_input_ids) > self.max_length:
            full_input_ids = full_input_ids[: self.max_length]

        labels = full_input_ids[1:] + [self.tokenizer.pad_token_id]

        full_input_ids = full_input_ids + [self.tokenizer.pad_token_id] * (
            self.max_length - len(full_input_ids)
        )
        labels = labels + [-100] * (self.max_length - len(labels))

        attention_mask = [
            1 if id != self.tokenizer.pad_token_id else 0 for id in full_input_ids
        ]

        return {
            "input_ids": torch.tensor(full_input_ids, dtype=torch.long),
            "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
            "labels": torch.tensor(labels, dtype=torch.long),
        }


class ChessInstructStockfishDataset(Dataset):
    def __init__(
        self, samples: List[Dict], tokenizer: ChessTokenizer, max_length: int = 512
    ):
        self.samples = samples
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]
        moves = sample["moves"]
        black_move = sample["black_move"]
        legal_moves = sample["legal_moves"]

        input_ids = self.tokenizer.encode_instruct(moves[:-1], include_last_black=True)

        black_move_token_id = self.tokenizer.token_to_id.get(
            black_move, self.tokenizer.unk_token_id
        )

        if len(input_ids) > self.max_length - 1:
            input_ids = input_ids[: self.max_length - 1]

        input_ids = input_ids + [self.tokenizer.pad_token_id] * (
            self.max_length - len(input_ids)
        )
        attention_mask = [
            1 if id != self.tokenizer.pad_token_id else 0 for id in input_ids
        ]

        return {
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
            "labels": torch.tensor(black_move_token_id, dtype=torch.long),
            "moves": moves[:-1],
            "black_move": black_move,
            "legal_moves": legal_moves,
        }


def collate_pretrain(batch):
    return {
        "input_ids": torch.stack([item["input_ids"] for item in batch]),
        "attention_mask": torch.stack([item["attention_mask"] for item in batch]),
        "labels": torch.stack([item["labels"] for item in batch]),
    }


def collate_instruct_supervised(batch):
    return {
        "input_ids": torch.stack([item["input_ids"] for item in batch]),
        "attention_mask": torch.stack([item["attention_mask"] for item in batch]),
        "labels": torch.stack([item["labels"] for item in batch]),
    }


def collate_instruct_stockfish(batch):
    return {
        "input_ids": torch.stack([item["input_ids"] for item in batch]),
        "attention_mask": torch.stack([item["attention_mask"] for item in batch]),
        "labels": torch.stack([item["labels"] for item in batch]),
        "moves": [item["moves"] for item in batch],
        "black_moves": [item["black_move"] for item in batch],
        "legal_moves": [item["legal_moves"] for item in batch],
    }


print("Dataset classes defined")

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
from transformers import LlamaConfig, LlamaForCausalLM


class ChessTransformer(nn.Module):
    def __init__(
        self, config: LlamaConfig, vocab_size: int, instruct_mode: bool = False
    ):
        super().__init__()

        config.vocab_size = vocab_size
        self.config = config
        self.instruct_mode = instruct_mode

        self.model = LlamaForCausalLM(config)

        if instruct_mode:
            self.move_classifier = nn.Linear(config.hidden_size, vocab_size)

    def forward(self, input_ids, attention_mask=None, labels=None):
        if self.instruct_mode and labels is not None and labels.dim() == 1:
            outputs = self.model.model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                output_hidden_states=True,
            )

            hidden_states = outputs.last_hidden_state
            assert attention_mask is not None
            sequence_lengths = attention_mask.sum(dim=1) - 1
            batch_indices = torch.arange(
                hidden_states.size(0), device=hidden_states.device
            )
            last_hidden = hidden_states[batch_indices, sequence_lengths]

            logits = self.move_classifier(last_hidden)

            loss = None
            if labels is not None:
                loss = F.cross_entropy(logits, labels)

            return {"loss": loss, "logits": logits}
        else:
            outputs = self.model(
                input_ids=input_ids, attention_mask=attention_mask, labels=labels
            )

            return {"loss": outputs.loss, "logits": outputs.logits}


print("Model architecture defined")

In [ ]:
from collections import OrderedDict

import chess
import chess.pgn

import math


class StockfishEvaluator:
    def __init__(
        self,
        stockfish_path: str,
        depth: int = 15,
        temperature: float = 100.0,
        invalid_penalty: float = 10.0,
        cache_size: int = 100000,
    ):
        self.stockfish_path = stockfish_path
        self.depth = depth
        self.temperature = temperature
        self.invalid_penalty = invalid_penalty
        self.cache = OrderedDict()
        self.cache_size = cache_size
        self.engine = None

    def __enter__(self):
        try:
            self.engine = chess.engine.SimpleEngine.popen_uci(self.stockfish_path)
            print(f"Stockfish initialized at {self.stockfish_path}")
        except Exception as e:
            print(f"Warning: Could not initialize Stockfish: {e}")
            self.engine = None
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        if self.engine:
            self.engine.quit()

    def reconstruct_board(self, moves: List[str]) -> chess.Board:
        board = chess.Board()
        for move_uci in moves:
            try:
                move = chess.Move.from_uci(move_uci)
                if move in board.legal_moves:
                    board.push(move)
                else:
                    return None
            except:
                return None
        return board

    def get_move_scores(self, board: chess.Board) -> Dict[str, float]:
        if not self.engine:
            return {}

        position_key = board.fen()

        if position_key in self.cache:
            return self.cache[position_key]

        legal_moves = list(board.legal_moves)
        scores = {}

        for move in legal_moves:
            temp_board = board.copy()
            temp_board.push(move)

            try:
                info = self.engine.analyse(
                    temp_board, chess.engine.Limit(depth=self.depth)
                )
                score = info["score"].relative.score(mate_score=10000)
                scores[move.uci()] = -score
            except:
                scores[move.uci()] = 0

        if len(self.cache) >= self.cache_size:
            self.cache.popitem(last=False)

        self.cache[position_key] = scores

        return scores

    def compute_stockfish_loss(
        self,
        logits: torch.Tensor,
        moves: List[str],
        legal_moves: List[str],
        tokenizer: ChessTokenizer,
    ) -> torch.Tensor:
        board = self.reconstruct_board(moves)

        if board is None:
            return torch.tensor(self.invalid_penalty, device=logits.device)

        probs = F.softmax(logits, dim=-1)
        predicted_idx = torch.argmax(probs).item()
        predicted_token = tokenizer.id_to_token.get(predicted_idx, "<UNK>")
        predicted_move = tokenizer.decode_move(predicted_token)

        if predicted_move not in legal_moves:
            return torch.tensor(self.invalid_penalty, device=logits.device)

        move_scores = self.get_move_scores(board)

        if not move_scores:
            return torch.tensor(0.0, device=logits.device)

        best_score = max(move_scores.values())

        stockfish_probs = {}
        total = 0.0

        for move_uci, score in move_scores.items():
            prob = math.exp((score - best_score) / self.temperature)
            stockfish_probs[move_uci] = prob
            total += prob

        for move_uci in stockfish_probs:
            stockfish_probs[move_uci] /= total

        target_dist = torch.zeros_like(logits)

        for move_uci, prob in stockfish_probs.items():
            token_id = tokenizer.token_to_id.get(move_uci, tokenizer.unk_token_id)
            if token_id < target_dist.size(-1):
                target_dist[token_id] = prob

        model_log_probs = F.log_softmax(logits, dim=-1)
        loss = F.kl_div(model_log_probs, target_dist, reduction="batchmean")

        return loss


print("Stockfish evaluator defined")

In [ ]:
import pandas as pd
import io


def load_dataset(csv_path, max_games=None):
    print(f"Loading dataset from {csv_path}...")
    df = pd.read_csv(csv_path)

    print(f"Total games in CSV: {len(df)}")
    print(f"Columns: {df.columns.tolist()}")

    if max_games:
        df = df.head(max_games)
        print(f"Limited to {len(df)} games")

    return df


def process_games_pretrain(
    df: pd.DataFrame, max_games: Optional[int] = None
) -> List[Dict]:
    samples = []

    games_to_process = len(df) if max_games is None else min(max_games, len(df))

    print(f"Processing {games_to_process} games for pretraining...")

    for idx in range(games_to_process):
        if idx % 1000 == 0:
            print(f"Processed {idx}/{games_to_process} games")

        try:
            pgn_text = df.iloc[idx]["pgn"]

            game = chess.pgn.read_game(io.StringIO(pgn_text))
            if game is None:
                continue

            moves = [move.uci() for move in game.mainline_moves()]

            if len(moves) >= 10:
                samples.append({"moves": moves})

        except Exception as e:
            continue

    print(f"Extracted {len(samples)} games for pretraining")
    return samples


def process_games_instruct(
    df: pd.DataFrame, max_games: Optional[int] = None
) -> List[Dict]:
    samples = []

    games_to_process = len(df) if max_games is None else min(max_games, len(df))

    print(f"Processing {games_to_process} games for instruction tuning...")

    for idx in range(games_to_process):
        if idx % 1000 == 0:
            print(f"Processed {idx}/{games_to_process} games, {len(samples)} positions")

        try:
            pgn_text = df.iloc[idx]["pgn"]

            game = chess.pgn.read_game(io.StringIO(pgn_text))
            if game is None:
                continue

            moves_list = [move.uci() for move in game.mainline_moves()]

            if len(moves_list) < 2:
                continue

            for i in range(1, len(moves_list), 2):
                if i >= len(moves_list):
                    break

                board_temp = chess.Board()
                for m in moves_list[:i]:
                    board_temp.push_uci(m)

                legal_moves = [m.uci() for m in board_temp.legal_moves]

                samples.append(
                    {
                        "moves": moves_list[: i + 1],
                        "black_move": moves_list[i],
                        "legal_moves": legal_moves,
                    }
                )

        except Exception as e:
            continue

    print(f"Extracted {len(samples)} positions for instruction tuning")
    return samples


print("Data processing functions defined")

In [ ]:
from typing import Tuple
from torch.utils.data import DataLoader
import random


def create_pretrain_loaders(
    samples: List[Dict],
    tokenizer: ChessTokenizer,
    config: TrainingConfig,
) -> Tuple[DataLoader, DataLoader]:
    random.shuffle(samples)
    split_idx = int(config.train_split * len(samples))
    train_samples = samples[:split_idx]
    val_samples = samples[split_idx:]

    print(f"Pretrain - Train: {len(train_samples)}, Val: {len(val_samples)}")

    train_dataset = ChessPretrainDataset(train_samples, tokenizer, config.max_length)
    val_dataset = ChessPretrainDataset(val_samples, tokenizer, config.max_length)

    train_loader = DataLoader(
        train_dataset,
        batch_size=config.batch_size,
        shuffle=True,
        collate_fn=collate_pretrain,
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=config.batch_size,
        shuffle=False,
        collate_fn=collate_pretrain,
    )

    return train_loader, val_loader


def create_instruct_supervised_loaders(
    samples: List[Dict],
    tokenizer: ChessTokenizer,
    config: TrainingConfig,
) -> Tuple[DataLoader, DataLoader]:
    random.shuffle(samples)
    split_idx = int(config.train_split * len(samples))
    train_samples = samples[:split_idx]
    val_samples = samples[split_idx:]

    print(f"Instruct Supervised - Train: {len(train_samples)}, Val: {len(val_samples)}")

    train_dataset = ChessInstructSupervisedDataset(
        train_samples, tokenizer, config.max_length
    )
    val_dataset = ChessInstructSupervisedDataset(
        val_samples, tokenizer, config.max_length
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=config.batch_size,
        shuffle=True,
        collate_fn=collate_instruct_supervised,
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=config.batch_size,
        shuffle=False,
        collate_fn=collate_instruct_supervised,
    )

    return train_loader, val_loader


def create_instruct_stockfish_loaders(
    samples: List[Dict],
    tokenizer: ChessTokenizer,
    config: TrainingConfig,
) -> Tuple[DataLoader, DataLoader]:
    random.shuffle(samples)
    split_idx = int(config.train_split * len(samples))
    train_samples = samples[:split_idx]
    val_samples = samples[split_idx:]

    print(f"Instruct Stockfish - Train: {len(train_samples)}, Val: {len(val_samples)}")

    train_dataset = ChessInstructStockfishDataset(
        train_samples, tokenizer, config.max_length
    )
    val_dataset = ChessInstructStockfishDataset(
        val_samples, tokenizer, config.max_length
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=config.batch_size,
        shuffle=True,
        collate_fn=collate_instruct_stockfish,
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=config.batch_size,
        shuffle=False,
        collate_fn=collate_instruct_stockfish,
    )

    return train_loader, val_loader


print("DataLoader creation functions defined")

In [ ]:
def create_model(
    tokenizer: ChessTokenizer,
    config: TrainingConfig,
    device: torch.device,
    instruct_mode: bool = False,
) -> ChessTransformer:
    print(f"Initializing model (instruct_mode={instruct_mode})...")

    model_config = LlamaConfig(
        vocab_size=len(tokenizer),
        hidden_size=config.hidden_size,
        intermediate_size=config.intermediate_size,
        num_hidden_layers=config.num_hidden_layers,
        num_attention_heads=config.num_attention_heads,
        max_position_embeddings=config.max_position_embeddings,
        rms_norm_eps=1e-5,
        initializer_range=0.02,
        use_cache=False,
        pad_token_id=tokenizer.pad_token_id,
        bos_token_id=tokenizer.bos_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

    model = ChessTransformer(model_config, len(tokenizer), instruct_mode=instruct_mode)
    model = model.to(device)

    if torch.cuda.device_count() > 1:
        print(f"Using {torch.cuda.device_count()} GPUs with DataParallel")
        model = nn.DataParallel(model)

    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

    print(f"Total parameters: {total_params:,}")
    print(f"Trainable parameters: {trainable_params:,}")

    return model


def save_checkpoint(
    model: ChessTransformer,
    tokenizer: ChessTokenizer,
    config: TrainingConfig,
    stage: str,
    epoch: int,
    metrics: Dict[str, float],
):
    save_path = os.path.join(config.save_dir, f"{stage}_epoch_{epoch}")
    os.makedirs(save_path, exist_ok=True)

    model_to_save = model.module if isinstance(model, nn.DataParallel) else model

    model_to_save.model.save_pretrained(save_path, safe_serialization=True)

    tokenizer.save_pretrained(save_path)

    with open(os.path.join(save_path, "training_metadata.json"), "w") as f:
        json.dump(
            {
                "epoch": epoch,
                "stage": stage,
                "metrics": metrics,
            },
            f,
            indent=2,
        )

    print(f"Checkpoint saved to {save_path}")


print("Model utilities defined")

In [ ]:
def train_epoch_pretrain(model, dataloader, optimizer, device, config):
    model.train()
    total_loss = 0.0

    for batch_idx, batch in enumerate(dataloader):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad()
        outputs = model(
            input_ids=input_ids, attention_mask=attention_mask, labels=labels
        )
        loss = outputs["loss"]

        if isinstance(model, nn.DataParallel):
            loss = loss.mean()

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item()

        if batch_idx % config.log_every_n_batches == 0:
            print(f"Batch {batch_idx}/{len(dataloader)}: Loss={loss.item():.4f}")

    return {"loss": total_loss / len(dataloader)}


def evaluate_pretrain(model, dataloader, device):
    model.eval()
    total_loss = 0.0

    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(
                input_ids=input_ids, attention_mask=attention_mask, labels=labels
            )
            loss = outputs["loss"]

            if isinstance(model, nn.DataParallel):
                loss = loss.mean()

            total_loss += loss.item()

    return {"loss": total_loss / len(dataloader)}


def train_epoch_instruct_supervised(model, dataloader, optimizer, device, config):
    model.train()
    total_loss = 0.0

    for batch_idx, batch in enumerate(dataloader):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad()
        outputs = model(
            input_ids=input_ids, attention_mask=attention_mask, labels=labels
        )
        loss = outputs["loss"]

        if isinstance(model, nn.DataParallel):
            loss = loss.mean()

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item()

        if batch_idx % config.log_every_n_batches == 0:
            print(f"Batch {batch_idx}/{len(dataloader)}: Loss={loss.item():.4f}")

    return {"loss": total_loss / len(dataloader)}


def evaluate_instruct_supervised(model, dataloader, device):
    model.eval()
    total_loss = 0.0

    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(
                input_ids=input_ids, attention_mask=attention_mask, labels=labels
            )
            loss = outputs["loss"]

            if isinstance(model, nn.DataParallel):
                loss = loss.mean()

            total_loss += loss.item()

    return {"loss": total_loss / len(dataloader)}


def train_epoch_instruct_stockfish(
    model,
    dataloader,
    optimizer,
    device,
    tokenizer,
    config,
    stockfish_evaluator,
    alpha=0.5,
):
    model.train()
    total_loss = 0.0
    total_ce_loss = 0.0
    total_sf_loss = 0.0
    total_correct = 0
    total_legal = 0
    total_samples = 0

    for batch_idx, batch in enumerate(dataloader):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad()

        outputs = model(
            input_ids=input_ids, attention_mask=attention_mask, labels=labels
        )
        ce_loss = outputs["loss"]
        logits = outputs["logits"]

        if isinstance(model, nn.DataParallel):
            ce_loss = ce_loss.mean()

        sf_loss = torch.tensor(0.0, device=device)

        if batch_idx % config.use_stockfish_every_n_batches == 0:
            batch_sf_losses = []

            for i in range(len(batch["moves"])):
                try:
                    sf_loss_i = stockfish_evaluator.compute_stockfish_loss(
                        logits[i], batch["moves"][i], batch["legal_moves"][i], tokenizer
                    )
                    batch_sf_losses.append(sf_loss_i)
                except:
                    batch_sf_losses.append(torch.tensor(0.0, device=device))

            if batch_sf_losses:
                sf_loss = torch.stack(batch_sf_losses).mean()

        loss = (1 - alpha) * ce_loss + alpha * sf_loss

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item()
        total_ce_loss += ce_loss.item()
        total_sf_loss += sf_loss.item() if isinstance(sf_loss, torch.Tensor) else 0.0

        predictions = torch.argmax(logits, dim=-1)
        total_correct += (predictions == labels).sum().item()

        for i in range(len(predictions)):
            pred_token = tokenizer.id_to_token.get(predictions[i].item(), "<UNK>")
            pred_move = tokenizer.decode_move(pred_token)
            if pred_move in batch["legal_moves"][i]:
                total_legal += 1

        total_samples += labels.size(0)

        if batch_idx % config.log_every_n_batches == 0:
            print(
                f"Batch {batch_idx}/{len(dataloader)}: Loss={loss.item():.4f}, CE={ce_loss.item():.4f}, "
                f"SF={sf_loss.item() if isinstance(sf_loss, torch.Tensor) else 0:.4f}, "
                f"Acc={total_correct/total_samples:.4f}, Legal={total_legal/total_samples:.4f}"
            )

    return {
        "loss": total_loss / len(dataloader),
        "ce_loss": total_ce_loss / len(dataloader),
        "sf_loss": total_sf_loss / len(dataloader),
        "accuracy": total_correct / total_samples,
        "legal_move_rate": total_legal / total_samples,
    }


def evaluate_instruct_stockfish(model, dataloader, device, tokenizer):
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total_legal = 0
    total_samples = 0

    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(
                input_ids=input_ids, attention_mask=attention_mask, labels=labels
            )
            loss = outputs["loss"]
            logits = outputs["logits"]

            if isinstance(model, nn.DataParallel):
                loss = loss.mean()

            total_loss += loss.item()

            predictions = torch.argmax(logits, dim=-1)
            total_correct += (predictions == labels).sum().item()

            for i in range(len(predictions)):
                pred_token = tokenizer.id_to_token.get(predictions[i].item(), "<UNK>")
                pred_move = tokenizer.decode_move(pred_token)
                if pred_move in batch["legal_moves"][i]:
                    total_legal += 1

            total_samples += labels.size(0)

    return {
        "loss": total_loss / len(dataloader),
        "accuracy": total_correct / total_samples,
        "legal_move_rate": total_legal / total_samples,
    }


print("Training and evaluation functions defined")

In [ ]:
def test_model_pretrain(model, tokenizer, device):
    print("\n" + "=" * 50)
    print("Testing Pretrain Model")
    print("=" * 50)

    model_to_test = model.module if isinstance(model, nn.DataParallel) else model
    model_to_test.eval()

    test_moves = ["e2e4", "e7e5", "g1f3"]
    input_ids = tokenizer.encode_pretrain(test_moves)
    input_ids = torch.tensor(input_ids, dtype=torch.long).unsqueeze(0).to(device)
    attention_mask = torch.ones_like(input_ids)

    with torch.no_grad():
        outputs = model_to_test(input_ids, attention_mask)
        logits = outputs["logits"]
        last_token_logits = logits[0, -1, :]
        top_k = torch.topk(last_token_logits, k=10)

        print(f"\nGiven moves: {' '.join(test_moves)}")
        print("Top 10 predicted next moves:")

        for i in range(10):
            score = top_k.values[i].item()
            idx = top_k.indices[i].item()
            token = tokenizer.id_to_token.get(idx, "<UNK>")
            print(f"{i+1}. {token:15} (score: {score:7.4f})")


def test_model_instruct(model, tokenizer, device):
    print("\n" + "=" * 50)
    print("Testing Instruct Model")
    print("=" * 50)

    model_to_test = model.module if isinstance(model, nn.DataParallel) else model
    model_to_test.eval()

    test_moves = ["e2e4"]

    input_ids = tokenizer.encode_instruct(test_moves, include_last_black=True)
    input_ids = torch.tensor(input_ids, dtype=torch.long).unsqueeze(0).to(device)
    attention_mask = torch.ones_like(input_ids)

    board = chess.Board()
    board.push_uci("e2e4")
    legal_moves = [move.uci() for move in board.legal_moves]

    with torch.no_grad():
        outputs = model_to_test(input_ids, attention_mask)
        logits = outputs["logits"]

        sequence_length = attention_mask.sum(dim=1) - 1
        last_logits = logits[0, sequence_length[0], :]

        top_k = torch.topk(last_logits, k=10)

        print(f"\nWhite plays: e2e4")
        print("Top 10 predicted black responses:")

        for i in range(10):
            score = top_k.values[i].item()
            idx = top_k.indices[i].item()
            token = tokenizer.id_to_token.get(idx, "<UNK>")
            move = tokenizer.decode_move(token)
            legal = "Legal" if move in legal_moves else "Illegal"
            print(f"{i+1}. {token:15} {legal} (score: {score:7.4f})")


print("Testing functions defined")

In [ ]:
df = load_dataset(
    csv_path="/kaggle/input/chesscom-user-games-60000-games/club_games_data.csv",
    max_games=config.max_games,
)
print(f"\nDataset shape: {df.shape}")
if "pgn" in df.columns:
    print(f"First game preview:\n{df['pgn'].iloc[0][:200]}...")

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"Available GPUs: {torch.cuda.device_count()}")

print("\n" + "=" * 50)
print("PHASE 1: PRETRAINING")
print("=" * 50)

pretrain_samples = process_games_pretrain(df, max_games=config.max_games)
pretrain_train_loader, pretrain_val_loader = create_pretrain_loaders(
    pretrain_samples, pretrain_tokenizer, config
)

pretrain_model = create_model(pretrain_tokenizer, config, device, instruct_mode=False)

optimizer = torch.optim.AdamW(
    pretrain_model.parameters(),
    lr=config.learning_rate,
    weight_decay=config.weight_decay,
)

for epoch in range(config.pretrain_epochs):
    print(f"\nEpoch {epoch + 1}/{config.pretrain_epochs}")
    print("-" * 50)

    train_metrics = train_epoch_pretrain(
        pretrain_model, pretrain_train_loader, optimizer, device, config
    )
    print(f"Train Loss: {train_metrics['loss']:.4f}")

    val_metrics = evaluate_pretrain(pretrain_model, pretrain_val_loader, device)
    print(f"Val Loss: {val_metrics['loss']:.4f}")

    save_checkpoint(
        pretrain_model, pretrain_tokenizer, config, "pretrain", epoch + 1, val_metrics
    )

print("\nPretraining Complete!")

test_model_pretrain(pretrain_model, pretrain_tokenizer, device)

pretrain_save_path = os.path.join(config.save_dir, "pretrain_final")
os.makedirs(pretrain_save_path, exist_ok=True)
model_to_save = (
    pretrain_model.module
    if isinstance(pretrain_model, nn.DataParallel)
    else pretrain_model
)
torch.save(model_to_save.state_dict(), os.path.join(pretrain_save_path, "model.pt"))
pretrain_tokenizer.save_pretrained(pretrain_save_path)
print(f"\nPretrained model saved to {pretrain_save_path}")

print("\n" + "=" * 50)
print("PHASE 2A: INSTRUCTION SUPERVISED")
print("=" * 50)

instruct_samples = process_games_instruct(df, max_games=config.max_games)
instruct_supervised_train_loader, instruct_supervised_val_loader = (
    create_instruct_supervised_loaders(instruct_samples, instruct_tokenizer, config)
)

instruct_model = create_model(instruct_tokenizer, config, device, instruct_mode=True)

print("\nTransferring pretrained weights...")
pretrain_state = (
    pretrain_model.module.state_dict()
    if isinstance(pretrain_model, nn.DataParallel)
    else pretrain_model.state_dict()
)
instruct_state = (
    instruct_model.module.state_dict()
    if isinstance(instruct_model, nn.DataParallel)
    else instruct_model.state_dict()
)

for name, param in pretrain_state.items():
    if name in instruct_state and instruct_state[name].shape == param.shape:
        instruct_state[name] = param

if isinstance(instruct_model, nn.DataParallel):
    instruct_model.module.load_state_dict(instruct_state, strict=False)
else:
    instruct_model.load_state_dict(instruct_state, strict=False)
print("Weight transfer complete!")

optimizer = torch.optim.AdamW(
    instruct_model.parameters(),
    lr=config.learning_rate * 0.1,
    weight_decay=config.weight_decay,
)

for epoch in range(config.instruct_supervised_epochs):
    print(f"\nEpoch {epoch + 1}/{config.instruct_supervised_epochs}")
    print("-" * 50)

    train_metrics = train_epoch_instruct_supervised(
        instruct_model,
        instruct_supervised_train_loader,
        optimizer,
        device,
        config,
    )
    print(f"Train Loss: {train_metrics['loss']:.4f}")

    val_metrics = evaluate_instruct_supervised(
        instruct_model, instruct_supervised_val_loader, device
    )
    print(f"Val Loss: {val_metrics['loss']:.4f}")

    save_checkpoint(
        instruct_model,
        instruct_tokenizer,
        config,
        "instruct_supervised",
        epoch + 1,
        val_metrics,
    )

print("\nInstruction Supervised Learning Complete!")

print("\n" + "=" * 50)
print("PHASE 2B: STOCKFISH-GUIDED")
print("=" * 50)

instruct_stockfish_train_loader, instruct_stockfish_val_loader = (
    create_instruct_stockfish_loaders(instruct_samples, instruct_tokenizer, config)
)

use_stockfish = os.path.exists(config.stockfish_path)
if not use_stockfish:
    print(f"Warning: Stockfish not found at {config.stockfish_path}")
    print("Skipping Stockfish phase")
else:
    stockfish_evaluator = StockfishEvaluator(
        config.stockfish_path,
        config.stockfish_depth,
        config.stockfish_temperature,
        config.invalid_move_penalty,
    )

    with stockfish_evaluator:
        optimizer = torch.optim.AdamW(
            instruct_model.parameters(),
            lr=config.learning_rate * 0.01,
            weight_decay=config.weight_decay,
        )

        for epoch in range(config.instruct_stockfish_epochs):
            print(f"\nEpoch {epoch + 1}/{config.instruct_stockfish_epochs}")
            print("-" * 50)

            train_metrics = train_epoch_instruct_stockfish(
                instruct_model,
                instruct_stockfish_train_loader,
                optimizer,
                device,
                instruct_tokenizer,
                config,
                stockfish_evaluator,
                config.stockfish_alpha,
            )

            print(f"Train Loss: {train_metrics['loss']:.4f}")
            print(f"  CE Loss: {train_metrics['ce_loss']:.4f}")
            print(f"  SF Loss: {train_metrics['sf_loss']:.4f}")
            print(f"  Accuracy: {train_metrics['accuracy']:.4f}")
            print(f"  Legal Rate: {train_metrics['legal_move_rate']:.4f}")

            val_metrics = evaluate_instruct_stockfish(
                instruct_model,
                instruct_stockfish_val_loader,
                device,
                instruct_tokenizer,
            )

            print(f"Val Loss: {val_metrics['loss']:.4f}")
            print(f"Val Accuracy: {val_metrics['accuracy']:.4f}")
            print(f"Val Legal Rate: {val_metrics['legal_move_rate']:.4f}")

            save_checkpoint(
                instruct_model,
                instruct_tokenizer,
                config,
                "instruct_stockfish",
                epoch + 1,
                val_metrics,
            )

print("\nStockfish-Guided Fine-tuning Complete!")

test_model_instruct(instruct_model, instruct_tokenizer, device)

instruct_save_path = os.path.join(config.save_dir, "instruct_final")
os.makedirs(instruct_save_path, exist_ok=True)
model_to_save = (
    instruct_model.module
    if isinstance(instruct_model, nn.DataParallel)
    else instruct_model
)
torch.save(model_to_save.state_dict(), os.path.join(instruct_save_path, "model.pt"))
instruct_tokenizer.save_pretrained(instruct_save_path)

print("\n" + "=" * 50)
print("TRAINING COMPLETE!")
print("=" * 50)
print(f"Pretrained model: {os.path.join(config.save_dir, 'pretrain_final')}")
print(f"Instruction model: {instruct_save_path}")